# Figure PS03 — Does protein length explain why phage genes sit lower in 3Di entropy?

**Status:** draft (Setonix). Answers agent-handoffs#38 (2026-11-02): test whether shorter phage proteins explain the bacteria-minus-phage gap in 3Di entropy of annotated ORFs (Results, *Phage genes sit lower in 3Di entropy than cellular genes*).

**Caption (draft).** Phage proteins are shorter than bacterial proteins, but length explains little of the 3Di-entropy gap at matched length.
**(a)** Cumulative length distributions of annotated (matched) stop-to-stop ORFs of 90–999 aa: bacteria (GTDB r232, 301.6 M ORFs), phage (Inphared, 2.64 M) and, for phage, the annotated CDS behind each matched ORF. Grey lines mark the length bins of the Results table.
**(b)** Mean 3Di entropy of matched ORFs by length: bacteria at each exact length; phage, all positives (filled) and positives whose CDS covers ≥ 80% of the ORF (open), as n-weighted means in 30 log-spaced length bins.
**(c)** Bacteria-minus-phage gap in each length bin: as published (bin means, open squares) and at exactly matched length, averaged over the phage (circles) or bacterial (diamonds) length distribution, for all phage positives (black) and for positives whose CDS covers ≥ 80% of the ORF (green). Bars are 95% intervals from 1,000 bootstrap resamples of 95% ANI clusters; the bacterial means are treated as fixed.

## Provenance

| item | value |
|---|---|
| plotting data | `../data/figPS03_{phage_length_3di,length_dist,gap_by_bin,percentiles}.tsv.gz`; inputs, sha256, seeds and denominators in `../data/figPS03_provenance.json` |
| produced by | `manuscript/prep/phage_length_gap_prepare_data.py` (Setonix; per-ORF data are not in git) |
| phage 3Di entropy | Gadi Inphared run (genome_entropy 0.3.0, encoding size 10,000, NVIDIA V100), the run behind Fig. S17 (`genome_entropy:General/Validation/Inphared/gadi/ge90aa/phg_*.tsv.gz`) |
| phage annotation | Setonix all-genome tables (#156): Pharokka v1.9.1 + phold 1.2.5 CDS, contig-edge flags, 3Di strings (for the number of states), 95% ANI clusters (skani) |
| bacteria | `../data/figP04_gtdb_length_3di.tsv.gz`: GTDB r232 matched ORFs, n / mean / quartiles per exact length (genome_entropy 0.2.0, encoding size 10,000, V100) |

## Definitions

- **Length** is the protein length of the stop-to-stop ORF, stop codon excluded; 3Di entropy is measured on that ORF. **CDS length** is the annotated gene's protein length (stop excluded).
- **Coverage** of a phage positive: length of the same-strand CDS that ends at the ORF's stop, divided by the ORF length.
- **Exact-length gap**: at each length ℓ with ≥ 10 phage ORFs, bacterial minus phage mean; averaged over ℓ with phage weights (bacteria evaluated at phage lengths), bacterial weights, or overlap weights n_p n_b / (n_p + n_b). Every length in every bin had ≥ 10 phage ORFs, so the common support is the whole bin.
- **Raw gap**: n-weighted bin means, as in the Results table.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

import msfig

msfig.style()
SLUG = "figPS03_phage_length_gap"
C = msfig.PALETTE

In [ ]:
t = msfig.data("figPS03_phage_length_3di.tsv.gz")
dist = msfig.data("figPS03_length_dist.tsv.gz").set_index("aa_length")
gap = msfig.data("figPS03_gap_by_bin.tsv.gz")
pct = msfig.data("figPS03_percentiles.tsv.gz")
g = msfig.data("figP04_gtdb_length_3di.tsv.gz")
bac = g[(g.dataset == "bacteria") & (g.cls == "matched")].set_index("aa_length").loc[90:999]
dist["bacteria_matched_orf"] = bac.n
BINS = ["90-149", "150-299", "300-599", "600-999"]
G = gap[gap.bacteria_ref == "gtdb_all"].set_index(["phage_class", "bin", "scheme"])


def est(cls, b, s):
    return round(G.loc[(cls, b, s), "estimate"], 2)


def qlen(s, x):
    c = s.cumsum() / s.sum()
    return int(c[c >= x].index[0])


# The numbers the text quotes.
assert dist.phage_matched_orf.sum() == 2641487 and dist.bacteria_matched_orf.sum() == 301622163
assert [qlen(dist.phage_matched_orf, q) for q in (.25, .5, .75)] == [128, 181, 298]
assert [qlen(dist.bacteria_matched_orf, q) for q in (.25, .5, .75)] == [208, 317, 455]
assert [qlen(dist.phage_matched_cds, 0.5)] == [178]
# raw reproduces the published table; exact-length (phage weights) is the adjusted gap
assert [est("all", b, "raw") for b in BINS] == [0.24, 0.34, 0.19, 0.09]
assert [est("all", b, "phage_w") for b in BINS] == [0.21, 0.29, 0.19, 0.09]
assert [est("all", b, "bac_w") for b in BINS] == [0.22, 0.29, 0.19, 0.11]
assert [est("cov80", b, "raw") for b in BINS] == [0.18, 0.24, 0.14, 0.08]
assert [est("cov80", b, "phage_w") for b in BINS] == [0.14, 0.19, 0.13, 0.08]
assert (est("all", "90-999", "raw"), est("all", "90-999", "phage_w")) == (0.46, 0.23)
assert (est("cov80", "90-999", "raw"), est("cov80", "90-999", "phage_w")) == (0.36, 0.15)
# contig-edge ORFs change nothing; the Setonix encoding gives the same gaps
for b in BINS + ["90-999"]:
    assert abs(G.loc[("complete", b, "phage_w"), "estimate"] - G.loc[("all", b, "phage_w"), "estimate"]) < 0.002
    assert abs(G.loc[("all_setonix_encoding", b, "phage_w"), "estimate"] - G.loc[("all", b, "phage_w"), "estimate"]) < 0.005
P = pct[(pct.bacteria_ref == "gtdb_all") & (pct.phage_class == "all") & (pct.bin == "90-999")].set_index("bacterial_quantile")
assert [round(P.loc[q, "phage_share_below"], 2) for q in ("q25", "median", "q75")] == [0.34, 0.67, 0.9]
A = t[t.cls == "all"].set_index("aa_length")
# finite-sample (Miller-Madow) bias bound: the largest possible difference if bacterial ORFs used all 20 states
bound = (20 - A.mean_k) / (2 * A.index * np.log(2))
assert round(bound.loc[90], 2) == 0.08 and round(bound.loc[200], 2) == 0.02 and bound.loc[400:].max() < 0.01

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(7.2, 2.5), gridspec_kw=dict(width_ratios=[1, 1.15, 1.05], wspace=0.62))
edges = [150, 300, 600]

# (a) cumulative length distributions
ax = axs[0]
for col, lab, c, ls in [("bacteria_matched_orf", "bacteria, ORF", C["blue"], "-"),
                        ("phage_matched_orf", "phage, ORF", C["vermillion"], "-"),
                        ("phage_matched_cds", "phage, CDS", C["vermillion"], ":")]:
    s = dist[col]
    ax.plot(s.index, s.cumsum() / s.sum(), color=c, ls=ls, lw=1.2, label=lab)
for e in edges:
    ax.axvline(e, color=C["grey"], lw=0.5, zorder=0)
ax.set_xscale("log")
ax.set_xticks([90, 150, 300, 600, 999], labels=["90", "150", "300", "600", "999"])
ax.minorticks_off()
ax.set_xlabel("protein length (aa)")
ax.set_ylabel("cumulative fraction")
ax.legend(loc="lower right", handlelength=1.6)

# (b) mean 3Di entropy at each exact length
ax = axs[1]
ax.plot(bac.index, bac["mean"], color=C["blue"], lw=1.0, label="bacteria")
# phage: n-weighted means in 30 log-spaced length bins (single lengths are noisy above 600 aa)
lb = np.unique(np.round(np.geomspace(90, 1000, 31)).astype(int))
for cls, lab, m, fc in [("all", "phage, all", "o", None), ("cov80", "phage, CDS ≥ 80%", "o", "white")]:
    x = t[t.cls == cls].assign(b=lambda d: pd.cut(d.aa_length, lb, right=False))
    s = x.groupby("b", observed=True).apply(lambda d: pd.Series({"L": np.average(d.aa_length, weights=d.n),
                                                                 "m": np.average(d["mean"], weights=d.n)}))
    ax.plot(s.L, s.m, color=C["vermillion"], lw=0.8, marker=m, ms=2.6, mfc=fc or C["vermillion"], mew=0.7, label=lab)
for e in edges:
    ax.axvline(e, color=C["grey"], lw=0.5, zorder=0)
ax.set_xscale("log")
ax.set_xticks([90, 150, 300, 600, 999], labels=["90", "150", "300", "600", "999"])
ax.minorticks_off()
ax.set_xlabel("ORF length (aa)")
ax.set_ylabel("mean 3Di entropy (bits)")
ax.set_ylim(1.9, 3.45)
ax.legend(loc="lower right", handlelength=1.4, fontsize=6, borderaxespad=0.2)

# (c) gap per bin: raw vs exact-length matched
ax = axs[2]
SCH = [("raw", "s", "white"), ("phage_w", "o", None), ("bac_w", "D", None)]
for j, (cls, col) in enumerate([("all", C["black"]), ("cov80", C["green"])]):
    for k, (s, m, fc) in enumerate(SCH):
        y = np.arange(len(BINS)) + (j * 3 + k - 2.5) * 0.12
        r = G.loc[[(cls, b, s) for b in BINS]]
        ax.errorbar(r.estimate, y, xerr=[r.estimate - r.ci_lo, r.ci_hi - r.estimate], fmt=m, ms=3.2,
                    color=col, mfc=fc or col, mew=0.8, lw=0.8, capsize=0)
ax.set_yticks(range(len(BINS)), labels=[b.replace("-", "–") for b in BINS])
ax.set_ylabel("length bin (aa)")
ax.invert_yaxis()
ax.axvline(0, color=C["grey"], lw=0.5)
ax.set_xlim(0, 0.4)
ax.set_xlabel("bacteria − phage, 3Di entropy (bits)")
h = [Line2D([], [], color=C["black"], lw=0.8, marker="s", mfc="white", ms=3.2, label="raw (bin means)"),
     Line2D([], [], color=C["black"], lw=0.8, marker="o", ms=3.2, label="exact length, phage wts"),
     Line2D([], [], color=C["black"], lw=0.8, marker="D", ms=3.2, label="exact length, bacterial wts"),
     Line2D([], [], color=C["black"], lw=2, label="all phage positives"),
     Line2D([], [], color=C["green"], lw=2, label="phage CDS ≥ 80% of ORF")]
ax.legend(handles=h, loc="upper center", bbox_to_anchor=(0.45, -0.25), ncol=2, fontsize=6, handlelength=1.6)

for ax, l in zip(axs, "abc"):
    ax.text(-0.2, 1.04, l, transform=ax.transAxes, fontweight="bold", fontsize=9)
msfig.save(fig, SLUG)

In [ ]:
# Tables quoted in the Results and in agent-handoffs#38
show = gap[(gap.bacteria_ref == "gtdb_all") & gap.phage_class.isin(["all", "cov80", "all_setonix_encoding"])]
show = show.assign(v=show.apply(lambda r: f"{r.estimate:.3f} [{r.ci_lo:.3f}, {r.ci_hi:.3f}]", axis=1))
print(show.pivot_table(index=["phage_class", "bin"], columns="scheme", values="v", aggfunc="first").to_string())
pp = pct[(pct.bacteria_ref == "gtdb_all") & pct.phage_class.isin(["all", "cov80"])]
print(pp.pivot_table(index=["phage_class", "bin"], columns="bacterial_quantile", values="phage_share_below").round(3).to_string())
print(pd.DataFrame({"mean_states": A.mean_k, "miller_madow": A.mm_correction, "bound_if_bacteria_used_20": bound}).loc[[90, 120, 200, 400, 800]].round(3).to_string())

## Notes

- **Length matters for the pooled comparison, little within bins.** Phage matched ORFs are shorter (median 181 aa against 317), and 3Di entropy rises with length, so pooling 90–999 aa doubles the gap (0.46 bits raw, 0.23 at exact length). The published table already splits by length; within its bins, matching exact length lowers the gap by 0.03–0.05 bits below 300 aa (≈ 15%) and not at all above.
- **Coverage and length are separate effects.** Restricting phage positives to CDS covering ≥ 80% of the ORF lowers the gap by 0.05–0.10 bits below 600 aa, and length matching then lowers it by up to a further 0.05 bits (below 300 aa); 0.14, 0.19, 0.13 and 0.08 bits remain. The bacterial positives are not yet restricted (Gadi, agent-handoffs#38); the prep script takes their restricted per-length table as `--bacteria name=path`.
- **The whole distribution is shifted.** At the same length, 34% of phage positives fall below the bacterial lower quartile, 67% below the bacterial median and 90% below the upper quartile (25, 50 and 75% if the distributions matched). The gap is not a tail of low-entropy phage ORFs.
- **Finite-sample bias cannot produce the gap.** The plug-in estimate is low by about (K − 1)/(2N ln 2) bits for K occupied states in N residues (Miller–Madow). At equal N the difference in bias between the two groups is ΔK/(2N ln 2). Phage positives use 9.8 states at 90 aa and 17.4 at 400 aa; even if every bacterial ORF used all 20, the bias difference would be at most 0.08 bits at 90 aa, 0.02 at 200 aa and < 0.01 above 400 aa, and it would make the *true* gap larger, because the higher-entropy group uses more states and is underestimated more.
- **Configuration.** Bacteria were encoded with genome_entropy 0.2.0, phage with 0.3.0, both at encoding size 10,000 on V100s. The Setonix phage encoding (≈ 0.2.0, encoding size 20,000, ROCm) gives the same gaps within 0.005 bits, consistent with the determinism test (class means move about 0.01 bits between configurations). A matched-configuration run of the bacteria and the phage has not been done.
- **Contig edges** (truncated ORFs) change no gap by more than 0.002 bits.
- **CDS versus ORF length.** Annotated phage CDS lengths are distributed like the matched ORF lengths (median 178 against 181 aa); the bacterial CDS length distribution has not been reduced yet (Gadi).